# Reproducing the diagonal metadata illustration

**Supplementary notebook for:**

**An Effect-Algebraic Framework for Mediative Truth--Falsity Semantics in Quantum Evidence Models**

**Author:** Oscar Montiel Ross

**License:** MIT License

Copyright (c) 2026 Oscar Montiel Ross

The code cells in this notebook are released under the MIT License. See `LICENSE.txt` in the supplementary repository for the full license text.

This notebook reproduces the numerical values reported in the manuscript's diagonal metadata illustration. The calculation is semantic and reproducibility-oriented. It does **not** train, validate, or benchmark an autonomous-driving perception model. It also does **not** define a detector, safety policy, decision rule, or claim quantum computational advantage.


## Input file

The notebook expects the following file in the same folder:

`zod_selected_preprocessed_rows.csv`

The file contains a minimal preprocessed representation of three selected ZOD-derived metadata rows. No raw ZOD sensor data, images, LiDAR, radar, video, trained perception models, or personally identifying information are included.


## Quantities reproduced

The contextual-complexity index is

$$c = 0.35c_{weather} + 0.25c_{light} + 0.20c_{road} + 0.20c_{density}.$$

For rows with a positive metadata risk indicator, distance is converted into

$$D = \operatorname{clip}(d/10,0,1), \qquad r = 1-D.$$

The illustrative truth--falsity pair is then

$$\mu_p = 0.55 + 0.35r + 0.10c,$$

$$\nu_p = 0.05 + 0.20D + 0.25c.$$

For rows without a positive metadata risk indicator,

$$\mu_p = 0.10 + 0.30c,$$

$$\nu_p = 0.60 + 0.25(1-c).$$

The diagnostic quantities are

$$\pi_p = \max\{0,1-\mu_p-\nu_p\}, \qquad \zeta_p = \max\{0,\mu_p+\nu_p-1\}.$$

The diagonal mediative value is

$$M_q(p,\rho)=(1-\pi_p-\zeta_p/2)\mu_p+(\pi_p+\zeta_p/2)(1-\nu_p).$$

In the manuscript, this is the diagonal reduction

$$M_q(p,\rho)=M(\mu_p,\nu_p).$$


In [1]:
# SPDX-License-Identifier: MIT
# Copyright (c) 2026 Oscar Montiel Ross
# Author: Oscar Montiel Ross
#
# This notebook accompanies the manuscript:
# An Effect-Algebraic Framework for Mediative Truth--Falsity Semantics
# in Quantum Evidence Models.
#
# The code cells are released under the MIT License. See LICENSE.txt.

from __future__ import annotations

import csv
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Dict, Optional

INPUT_FILE = Path("zod_selected_preprocessed_rows.csv")
OUTPUT_FILE = Path("zod_mfl_recomputed_results.csv")


In [2]:
def clip(value: float, lower: float = 0.0, upper: float = 1.0) -> float:
    """Clip a numeric value to the interval [lower, upper]."""
    return max(lower, min(upper, value))


def parse_float(value: str) -> Optional[float]:
    """Parse an optional floating-point value from a CSV field."""
    value = (value or "").strip()
    if value == "":
        return None
    return float(value)


def round3(value: Optional[float]) -> str:
    """
    Format a value to three decimals using half-up decimal rounding.
    """
    if value is None:
        return ""
    rounded = Decimal(str(value)).quantize(Decimal("0.001"), rounding=ROUND_HALF_UP)
    return f"{rounded:.3f}"


In [3]:
def compute_context_score(row: Dict[str, str]) -> float:
    """
    Compute c = 0.35*c_weather + 0.25*c_light + 0.20*c_road + 0.20*c_density.
    """
    c_weather = float(row["c_weather"])
    c_light = float(row["c_light"])
    c_road = float(row["c_road"])
    c_density = float(row["c_density"])
    return 0.35*c_weather + 0.25*c_light + 0.20*c_road + 0.20*c_density


def compute_truth_falsity(row: Dict[str, str], c: float) -> Dict[str, Optional[float]]:
    """
    Compute the illustrative truth--falsity pair and diagnostic quantities.
    """
    risk_indicator = int(row["risk_indicator"])
    distance_m = parse_float(row.get("distance_m", ""))

    if risk_indicator == 1:
        if distance_m is None:
            raise ValueError("Positive-risk rows require distance_m.")
        D = clip(distance_m / 10.0, 0.0, 1.0)
        r = 1.0 - D
        mu_p = 0.55 + 0.35*r + 0.10*c
        nu_p = 0.05 + 0.20*D + 0.25*c
    else:
        D = None
        r = None
        mu_p = 0.10 + 0.30*c
        nu_p = 0.60 + 0.25*(1.0 - c)

    pi_p = max(0.0, 1.0 - mu_p - nu_p)
    zeta_p = max(0.0, mu_p + nu_p - 1.0)
    M_q = (1.0 - pi_p - zeta_p/2.0)*mu_p + (pi_p + zeta_p/2.0)*(1.0 - nu_p)

    return {
        "D": D,
        "r": r,
        "mu_p": mu_p,
        "nu_p": nu_p,
        "pi_p": pi_p,
        "zeta_p": zeta_p,
        "M_q": M_q,
    }


def build_evidence_pattern(row: Dict[str, str]) -> str:
    """Build a compact evidence-pattern string for traceability."""
    fields = [
        row.get("reason_field", "").strip(),
        row.get("weather", "").strip(),
        row.get("time_of_day", "").strip(),
        row.get("road_condition", "").strip(),
    ]
    return "; ".join(field for field in fields if field)


In [4]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Cannot find {INPUT_FILE}. Place it in the notebook folder.")

rows_out = []

with INPUT_FILE.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        c = compute_context_score(row)
        computed = compute_truth_falsity(row, c)

        rows_out.append({
            "case_id": row["case_id"],
            "split": row["split"],
            "frame_id": row["frame_id"],
            "risk_indicator": row["risk_indicator"],
            "evidence_pattern": build_evidence_pattern(row),
            "context_score_c": round3(c),
            "D": round3(computed["D"]),
            "r": round3(computed["r"]),
            "mu_p": round3(computed["mu_p"]),
            "nu_p": round3(computed["nu_p"]),
            "pi_p": round3(computed["pi_p"]),
            "zeta_p": round3(computed["zeta_p"]),
            "M_q_p_rho": round3(computed["M_q"]),
        })

if not rows_out:
    raise ValueError(f"No rows were found in {INPUT_FILE}.")

rows_out


[{'case_id': '1',
  'split': 'val',
  'frame_id': '002182',
  'risk_indicator': '1',
  'evidence_pattern': 'Vehicle at 6.5m; cloudy; day; normal road',
  'context_score_c': '0.281',
  'D': '0.650',
  'r': '0.350',
  'mu_p': '0.701',
  'nu_p': '0.250',
  'pi_p': '0.049',
  'zeta_p': '0.000',
  'M_q_p_rho': '0.703'},
 {'case_id': '2',
  'split': 'val',
  'frame_id': '000325',
  'risk_indicator': '0',
  'evidence_pattern': 'No risk label; rain; night; wet road',
  'context_score_c': '0.815',
  'D': '',
  'r': '',
  'mu_p': '0.345',
  'nu_p': '0.646',
  'pi_p': '0.009',
  'zeta_p': '0.000',
  'M_q_p_rho': '0.345'},
 {'case_id': '3',
  'split': 'test',
  'frame_id': '002044',
  'risk_indicator': '1',
  'evidence_pattern': 'Vehicle at -2.3m; rain; day; wet road',
  'context_score_c': '0.608',
  'D': '0.000',
  'r': '1.000',
  'mu_p': '0.961',
  'nu_p': '0.202',
  'pi_p': '0.000',
  'zeta_p': '0.163',
  'M_q_p_rho': '0.948'}]

In [5]:
fieldnames = list(rows_out[0].keys())

with OUTPUT_FILE.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_out)

print(f"Wrote {OUTPUT_FILE}")
for row in rows_out:
    print(
        f"Case {row['case_id']}: "
        f"split/frame={row['split']}/{row['frame_id']}, "
        f"c={row['context_score_c']}, "
        f"(mu_p,nu_p)=({row['mu_p']},{row['nu_p']}), "
        f"(pi_p,zeta_p)=({row['pi_p']},{row['zeta_p']}), "
        f"M_q(p,rho)={row['M_q_p_rho']}"
    )


Wrote zod_mfl_recomputed_results.csv
Case 1: split/frame=val/002182, c=0.281, (mu_p,nu_p)=(0.701,0.250), (pi_p,zeta_p)=(0.049,0.000), M_q(p,rho)=0.703
Case 2: split/frame=val/000325, c=0.815, (mu_p,nu_p)=(0.345,0.646), (pi_p,zeta_p)=(0.009,0.000), M_q(p,rho)=0.345
Case 3: split/frame=test/002044, c=0.608, (mu_p,nu_p)=(0.961,0.202), (pi_p,zeta_p)=(0.000,0.163), M_q(p,rho)=0.948


## Expected rounded values

| Case | Split/frame | c | (mu_p, nu_p) | (pi_p, zeta_p) | M_q(p,rho) |
|---|---|---:|---:|---:|---:|
| 1 | val/002182 | 0.281 | (0.701, 0.250) | (0.049, 0.000) | 0.703 |
| 2 | val/000325 | 0.815 | (0.345, 0.646) | (0.009, 0.000) | 0.345 |
| 3 | test/002044 | 0.608 | (0.961, 0.202) | (0.000, 0.163) | 0.948 |


## Scope note

This notebook only reproduces the semantic quantities used in the diagonal metadata illustration. The metadata-derived values first define classical truth--falsity pairs, and the diagonal embedding realizes those pairs as Born expectations of quantum effects. Non-commuting effects are not required for this diagonal instantiation; they become relevant only when the positive and negative evidence channels are incompatible at the operator level.
